In [ ]:
import time
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.model_selection import train_test_split
from sklearn.model_selection import GridSearchCV
from sklearn.model_selection import RandomizedSearchCV
from sklearn.model_selection import cross_val_score
from sklearn.preprocessing import MinMaxScaler

from sklearn.naive_bayes import GaussianNB
from sklearn.naive_bayes import MultinomialNB
from sklearn.naive_bayes import BernoulliNB
from sklearn.neighbors import KNeighborsClassifier

from sklearn.metrics import accuracy_score
from sklearn.metrics import precision_score
from sklearn.metrics import recall_score
from sklearn.metrics import f1_score
from sklearn.metrics import roc_auc_score
from sklearn.metrics import confusion_matrix
from sklearn.metrics import ConfusionMatrixDisplay
from sklearn.metrics import RocCurveDisplay
from sklearn.metrics import PrecisionRecallDisplay

In [ ]:
df = pd.read_csv("spambase_csv.csv")
print("Dataset Loaded Successfully")

In [ ]:
df.head()

In [ ]:
print("Shape :", df.shape)

print("\nColumns")
print(df.columns)

print("\nData Types")
print(df.dtypes)

print("\nInformation")
df.info()

In [ ]:
df.describe()

In [ ]:
print(df.isnull().sum())

In [ ]:
print("Duplicate Rows :", df.duplicated().sum())

In [ ]:
df["class"].value_counts().plot(kind="bar")

plt.title("Class Distribution")
plt.xlabel("Class")
plt.ylabel("Count")
plt.show()

In [ ]:
plt.figure(figsize=(10,8))

corr = df.corr()

plt.imshow(corr, cmap="coolwarm")
plt.colorbar()

plt.xticks([])
plt.yticks([])

plt.title("Correlation Heatmap")
plt.show()

In [ ]:
df.hist(figsize=(15,12))

plt.tight_layout()
plt.show()

In [ ]:
plt.figure(figsize=(15,6))

df.boxplot(rot=90)

plt.title("Box Plot")
plt.show()

In [ ]:
X = df.drop("class", axis=1)
y = df["class"]

scaler = MinMaxScaler()
X = scaler.fit_transform(X)

print("Features Normalized")

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=42
)

print("Training Samples :", len(X_train))
print("Testing Samples :", len(X_test))

In [ ]:
gnb = GaussianNB()

start = time.time()
gnb.fit(X_train, y_train)
train_time_gnb = time.time() - start

start = time.time()
y_pred_gnb = gnb.predict(X_test)
prediction_time_gnb = time.time() - start

In [ ]:
mnb = MultinomialNB()

start = time.time()
mnb.fit(X_train, y_train)
train_time_mnb = time.time() - start

start = time.time()
y_pred_mnb = mnb.predict(X_test)
prediction_time_mnb = time.time() - start

In [ ]:
bnb = BernoulliNB()

start = time.time()
bnb.fit(X_train, y_train)
train_time_bnb = time.time() - start

start = time.time()
y_pred_bnb = bnb.predict(X_test)
prediction_time_bnb = time.time() - start

In [ ]:
knn = KNeighborsClassifier(n_neighbors=5)

start = time.time()
knn.fit(X_train, y_train)
train_time_knn = time.time() - start

start = time.time()
y_pred_knn = knn.predict(X_test)
prediction_time_knn = time.time() - start

In [ ]:
def evaluate(y_true, y_pred):
    accuracy = accuracy_score(y_true, y_pred)
    precision = precision_score(y_true, y_pred)
    recall = recall_score(y_true, y_pred)
    f1 = f1_score(y_true, y_pred)
    roc = roc_auc_score(y_true, y_pred)
    return accuracy, precision, recall, f1, roc

In [ ]:
g = evaluate(y_test, y_pred_gnb)
m = evaluate(y_test, y_pred_mnb)
b = evaluate(y_test, y_pred_bnb)

nb_results = pd.DataFrame({
    "Metric": ["Accuracy", "Precision", "Recall", "F1", "ROC-AUC"],
    "Gaussian": g,
    "Multinomial": m,
    "Bernoulli": b
})

nb_results

In [ ]:
ConfusionMatrixDisplay.from_predictions(y_test, y_pred_gnb)
plt.title("Gaussian Naive Bayes")
plt.show()

In [ ]:
ConfusionMatrixDisplay.from_predictions(y_test, y_pred_mnb)
plt.title("Multinomial Naive Bayes")
plt.show()

In [ ]:
ConfusionMatrixDisplay.from_predictions(y_test, y_pred_bnb)
plt.title("Bernoulli Naive Bayes")
plt.show()

In [ ]:
ConfusionMatrixDisplay.from_predictions(y_test, y_pred_knn)
plt.title("KNN")
plt.show()

In [ ]:
k_values = [1, 3, 5, 7, 9, 11]
results = []

for k in k_values:
    model = KNeighborsClassifier(n_neighbors=k)
    model.fit(X_train, y_train)
    pred = model.predict(X_test)

    accuracy = accuracy_score(y_test, pred)
    precision = precision_score(y_test, pred)
    recall = recall_score(y_test, pred)
    f1 = f1_score(y_test, pred)

    results.append([k, accuracy, precision, recall, f1])

knn_table = pd.DataFrame(
    results,
    columns=["k", "Accuracy", "Precision", "Recall", "F1"]
)

knn_table

In [ ]:
plt.figure(figsize=(6,4))

plt.plot(
    knn_table["k"],
    knn_table["Accuracy"],
    marker="o"
)

plt.title("Accuracy vs k")
plt.xlabel("k")
plt.ylabel("Accuracy")
plt.grid(True)
plt.show()

In [ ]:
parameters = {
    "n_neighbors": [1, 3, 5, 7, 9, 11],
    "weights": ["uniform", "distance"],
    "algorithm": ["auto", "kd_tree", "ball_tree"]
}

start = time.time()

grid_search = GridSearchCV(
    KNeighborsClassifier(),
    parameters,
    cv=5
)

grid_search.fit(X_train, y_train)
grid_time = time.time() - start

print("Best Parameters")
print(grid_search.best_params_)

print("\nBest CV Score")
print(grid_search.best_score_)

In [ ]:
random_search = RandomizedSearchCV(
    KNeighborsClassifier(),
    param_distributions=parameters,
    n_iter=10,
    cv=5,
    random_state=42
)

start = time.time()
random_search.fit(X_train, y_train)
random_time = time.time() - start

print("Best Parameters")
print(random_search.best_params_)

print("\nBest CV Score")
print(random_search.best_score_)

In [ ]:
search_result = pd.DataFrame({
    "Parameter": [
        "Best k",
        "Weights",
        "Algorithm",
        "CV Accuracy",
        "Execution Time"
    ],
    "GridSearchCV": [
        grid_search.best_params_["n_neighbors"],
        grid_search.best_params_["weights"],
        grid_search.best_params_["algorithm"],
        grid_search.best_score_,
        grid_time
    ],
    "RandomizedSearchCV": [
        random_search.best_params_["n_neighbors"],
        random_search.best_params_["weights"],
        random_search.best_params_["algorithm"],
        random_search.best_score_,
        random_time
    ]
})

search_result

In [ ]:
algorithms = ["kd_tree", "ball_tree"]
tree_result = []

for algo in algorithms:
    model = KNeighborsClassifier(
        n_neighbors=5,
        algorithm=algo
    )

    start = time.time()
    model.fit(X_train, y_train)
    train_time = time.time() - start

    start = time.time()
    pred = model.predict(X_test)
    predict_time = time.time() - start

    accuracy = accuracy_score(y_test, pred)

    tree_result.append(
        [algo, accuracy, train_time, predict_time]
    )

tree_table = pd.DataFrame(
    tree_result,
    columns=[
        "Algorithm",
        "Accuracy",
        "Training Time",
        "Prediction Time"
    ]
)

tree_table

In [ ]:
nb_cv = cross_val_score(GaussianNB(), X, y, cv=5)

best_knn = KNeighborsClassifier(
    n_neighbors=grid_search.best_params_["n_neighbors"],
    weights=grid_search.best_params_["weights"],
    algorithm=grid_search.best_params_["algorithm"]
)

knn_cv = cross_val_score(best_knn, X, y, cv=5)

cv_table = pd.DataFrame({
    "Fold": [1, 2, 3, 4, 5],
    "Naive Bayes": nb_cv,
    "Best KNN": knn_cv
})

cv_table

In [ ]:
average = pd.DataFrame({
    "Model": ["Naive Bayes", "Best KNN"],
    "Average Accuracy": [
        nb_cv.mean(),
        knn_cv.mean()
    ]
})

average

In [ ]:
RocCurveDisplay.from_estimator(gnb, X_test, y_test)
plt.title("Gaussian Naive Bayes ROC Curve")
plt.show()

RocCurveDisplay.from_estimator(knn, X_test, y_test)
plt.title("KNN ROC Curve")
plt.show()

In [ ]:
PrecisionRecallDisplay.from_estimator(gnb, X_test, y_test)
plt.title("Gaussian Naive Bayes Precision Recall Curve")
plt.show()

PrecisionRecallDisplay.from_estimator(knn, X_test, y_test)
plt.title("KNN Precision Recall Curve")
plt.show()

In [ ]:
training = pd.DataFrame({
    "Algorithm": [
        "Gaussian NB",
        "Multinomial NB",
        "Bernoulli NB",
        "Best KNN"
    ],
    "Training Time": [
        train_time_gnb,
        train_time_mnb,
        train_time_bnb,
        train_time_knn
    ]
})

training

In [ ]:
prediction = pd.DataFrame({
    "Algorithm": [
        "Gaussian NB",
        "Multinomial NB",
        "Bernoulli NB",
        "Best KNN"
    ],
    "Prediction Time": [
        prediction_time_gnb,
        prediction_time_mnb,
        prediction_time_bnb,
        prediction_time_knn
    ]
})

prediction

In [ ]:
plt.figure(figsize=(6,4))

plt.bar(training["Algorithm"], training["Training Time"])

plt.title("Training Time Comparison")
plt.ylabel("Time (seconds)")
plt.xticks(rotation=20)
plt.show()

In [ ]:
plt.figure(figsize=(6,4))

plt.bar(prediction["Algorithm"], prediction["Prediction Time"])

plt.title("Prediction Time Comparison")
plt.ylabel("Time (seconds)")
plt.xticks(rotation=20)
plt.show()

In [ ]:
classifier = pd.DataFrame({
    "Classifier": [
        "Gaussian NB",
        "Multinomial NB",
        "Bernoulli NB",
        "KNN"
    ],
    "Accuracy": [
        g[0],
        m[0],
        b[0],
        evaluate(y_test, y_pred_knn)[0]
    ]
})

classifier

In [ ]:
plt.figure(figsize=(6,4))

plt.bar(classifier["Classifier"], classifier["Accuracy"])

plt.title("Classifier Comparison")
plt.ylabel("Accuracy")
plt.xticks(rotation=20)
plt.show()